# Yield strength model for Alloy-718 type Ni-base superalloys

Physics-based yield strength (YS) model used to generate YS labels for **AlloyGPT-UQ**.
For each CALPHAD equilibrium configuration it combines:

| Contribution | Mechanism | Precipitate |
|---|---|---|
| $\sigma_{\gamma'}$ | Coherency strengthening (weakly coupled dislocations) | spherical $\gamma'$ (FCC_L12) |
| $\sigma_{\gamma''}$ | Coherency strengthening (strongly coupled dislocations) | oblate/disc $\gamma''$ (BCT_D022) |
| $\sigma_{ss}$ | Solid-solution strengthening | $\gamma$ matrix |
| $\sigma_0$ | Constant base term | — |

$$\sigma_y = \sigma_{\gamma'} + \sigma_{\gamma''} + \sigma_0 + \sigma_{ss}$$

### Data notice
The original inputs (phase fractions and phase compositions) were computed with **Thermo-Calc** and cannot be
redistributed under its license, so **no data is included in this repository**. The notebook runs in two modes:

* `USE_SYNTHETIC_DATA = True` (default): generates random placeholder inputs in the expected file format so the
  pipeline runs end to end. **Numbers produced in this mode are meaningless.**
* `USE_SYNTHETIC_DATA = False`: reads your own Thermo-Calc exports from `DATA_DIR` (format described below).

In [ ]:
import math
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import ScalarFormatter

## 1. Configuration

### Expected input format
`DATA_DIR` contains HDF5 files `batchconfig0.hdf5 ... batchconfig{N-1}.hdf5`. Each file holds the datasets below
(one row per alloy configuration). Rows from all files are concatenated.

| Key | Shape | Meaning |
|---|---|---|
| `comp` | (n, 7) | bulk alloy composition, columns `BULK_ELEMENTS` (Fe = balance) |
| `FCC_1`, `FCC_1_comp` | (n,), (n, 7) | $\gamma$ matrix: phase amount, phase composition |
| `FCC_2`, `FCC_2_comp` | (n,), (n, 7) | candidate $\gamma'$ phase |
| `FCC_3`, `FCC_3_comp` | (n,), (n, 7) | candidate $\gamma'$ phase |
| `BCT`, `BCT_comp` | (n,), (n, 7) | $\gamma''$ phase |

Phase compositions are indexed by `PHASE_ELEMENTS`, with the remainder (1 − Σ) assigned to Ni.

In [ ]:
USE_SYNTHETIC_DATA = True            # set False to run on your own Thermo-Calc exports
DATA_DIR = Path("data/phase_info")   # folder containing batchconfig{i}.hdf5
N_FILES = 120                        # number of batchconfig files to read
FILE_PATTERN = "batchconfig{i}.hdf5"

# Optional: experimental/literature YS values for comparison (CSV with one column named "YS_MPa").
LITERATURE_CSV = Path("data/literature_ys.csv")

# Optional: where to write the computed labels (leave as None to skip).
OUTPUT_PATH = None                   # e.g. Path("outputs/IN718_YS_labels.csv")

SEED = 0                             # synthetic-data seed

## 2. Model parameters

In [ ]:
# Element ordering
BULK_ELEMENTS  = ["Ni", "Cr", "Nb", "Mo", "Ti", "Al", "Co"]  # order of the `comp` columns
PHASE_ELEMENTS = ["Fe", "Cr", "Nb", "Mo", "Ti", "Al", "Co"]  # order of the `*_comp` columns (Ni = balance)

# Lattice-parameter coefficients (Angstrom) used in the Vegard-type mixing rule
LATTICE_PARAM = {
    "Ni": 3.73582,
    "Cr": 3.0186,
    "Fe": 3.0081,
    "Nb": 4.6844,
    "Al": 4.2877,
    "Ti": 3.1308,
    "Co": 2.5888,
    "Mo": 4.5508,
    "Mn": 3.535583,
    "Si": 3.038048,
    "Cu": 3.647662,
}

# Matrix properties
G   = 75.2e9      # shear modulus (Pa)
b   = 0.254e-9    # Burgers vector (m)
nu  = 0.33        # Poisson ratio
M   = 2.6         # Taylor factor
N_A = 6.022e23    # Avogadro number (1/mol)

# gamma' (spherical) geometry and misfit
R_GP      = 10e-9   # gamma' radius (m)
MISFIT_GP = 0.004   # lattice misfit

# gamma'' (oblate disc) geometry and misfit
A_GDP      = 50e-9    # disc radius (m)
C_GDP      = 10e-9    # disc half-thickness (m)
MISFIT_GDP = 0.0286   # tetragonal misfit
LOG_R_GDP  = 55.0     # outer/inner cut-off ratio in the gamma'' line-tension log term

# Solid-solution strengthening coefficients (MPa / at.frac^0.5), ordered as PHASE_ELEMENTS
K_SS = [153, 337, 1183, 1015, 775, 275, 225]

SIGMA_0 = 22.34   # constant base term (MPa)

## 3. Synthetic placeholder data

Random inputs in the same file format as the Thermo-Calc exports. These ranges are arbitrary placeholders
chosen only so the code runs; they are **not** CALPHAD results and carry no physical meaning.

In [ ]:
def make_synthetic_data(out_dir, n_files=4, n_per_file=2_000, seed=0):
    # Write random batchconfig{i}.hdf5 files with the expected keys and shapes.
    rng = np.random.default_rng(seed)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    def comp_block(n, lows, highs):
        return rng.uniform(lows, highs, size=(n, len(lows)))

    for i in range(n_files):
        n = n_per_file
        # phase amounts
        fcc1 = rng.uniform(0.70, 0.90, n)
        gp   = rng.uniform(0.01, 0.15, n)
        swap = rng.random(n) < 0.5                    # gamma' sometimes appears as FCC_2, sometimes FCC_3
        fcc2 = np.where(swap, gp, rng.uniform(0, 0.005, n))
        fcc3 = np.where(swap, rng.uniform(0, 0.005, n), gp)
        bct  = rng.uniform(0.01, 0.20, n)

        # compositions (placeholder ranges; 7 columns each)
        comp    = comp_block(n, [0.48, 0.17, 0.04, 0.02, 0.005, 0.002, 0.0], [0.56, 0.21, 0.06, 0.035, 0.012, 0.008, 0.01])
        mat     = comp_block(n, [0.15, 0.15, 0.00, 0.01, 0.000, 0.000, 0.0], [0.25, 0.25, 0.02, 0.04, 0.010, 0.010, 0.01])
        gp_comp = comp_block(n, [0.00, 0.00, 0.02, 0.00, 0.050, 0.050, 0.0], [0.05, 0.03, 0.08, 0.01, 0.120, 0.150, 0.01])
        bct_c   = comp_block(n, [0.00, 0.00, 0.15, 0.00, 0.000, 0.000, 0.0], [0.05, 0.03, 0.25, 0.02, 0.030, 0.010, 0.01])

        with h5py.File(out_dir / FILE_PATTERN.format(i=i), "w") as f:
            f["comp"] = comp
            f["FCC_1"], f["FCC_1_comp"] = fcc1, mat
            f["FCC_2"], f["FCC_2_comp"] = fcc2, gp_comp
            f["FCC_3"], f["FCC_3_comp"] = fcc3, gp_comp
            f["BCT"],   f["BCT_comp"]   = bct, bct_c
    return n_files


if USE_SYNTHETIC_DATA:
    DATA_DIR = Path("synthetic_data")
    N_FILES = make_synthetic_data(DATA_DIR, seed=SEED)
    print(f"Wrote {N_FILES} synthetic files to {DATA_DIR}/  (placeholder data, results are not physical)")

## 4. Load phase data

In [ ]:
def load_phase_data(data_dir, n_files, pattern=FILE_PATTERN):
    # Read and concatenate every dataset across batchconfig files.
    collected = {}
    for i in range(n_files):
        with h5py.File(Path(data_dir) / pattern.format(i=i), "r") as f:
            for key in f.keys():
                collected.setdefault(key, []).append(f[key][:])
    return {k: np.concatenate(v, axis=0) for k, v in collected.items()}


data = load_phase_data(DATA_DIR, N_FILES)
print("Keys:", list(data))
print("Configurations:", len(data["FCC_1"]))

In [ ]:
# gamma' is whichever of FCC_2 / FCC_3 has the larger phase amount
use_fcc2 = data["FCC_2"] > data["FCC_3"]

f_matrix = data["FCC_1"]                                              # gamma matrix amount
f_gp     = np.where(use_fcc2, data["FCC_2"], data["FCC_3"])           # gamma' amount
f_gdp    = data["BCT"]                                                # gamma'' amount

x_matrix = data["FCC_1_comp"]                                         # gamma matrix composition
x_gp     = np.where(use_fcc2[:, None], data["FCC_2_comp"], data["FCC_3_comp"])
x_gdp    = data["BCT_comp"]

## 5. Model

**Lattice parameter** of a phase (Ni as balance):
$a = \sum_k x_k a_k + \left(1 - \sum_k x_k\right) a_{Ni}$

**Molar volume** (4 atoms per FCC cell): $V_m = N_A\, a^3 / 4$, used to convert phase amounts to volume fractions.

**$\gamma'$, weakly coupled dislocations** with
$\bar r = \sqrt{2/3}\, r$, $\lambda = 2\bar r\left(\sqrt{\pi / 4f} - 1\right)$,
$\varepsilon = \tfrac{1}{3}\tfrac{1+\nu}{1-\nu}\delta$, line tension
$\Gamma = \tfrac{Gb^2}{4\pi}\, g(\theta)\, \ln\!\left(\tfrac{\lambda}{2b}\right)$:
$\tau_{edge} = 4.11\sqrt{\tfrac{G^3 b\, \varepsilon^3 \bar r^3}{\lambda^2 \Gamma_{edge}}}$,
$\tau_{screw} = 1.38\sqrt{\tfrac{G^3 b\, \varepsilon^3 \bar r^3}{\lambda^2 \Gamma_{screw}}}$,
$\sigma_{\gamma'} = M \max(\tau_{edge}, \tau_{screw})$.

**$\gamma''$, strongly coupled dislocations** with equivalent radius $r = (a^2 c)^{1/3}$, $\varepsilon = \tfrac{2}{3}\delta''$:
$\tau = k \tfrac{0.8}{\lambda}\left(\tfrac{\Gamma^3 \varepsilon G \bar r}{b^3}\right)^{1/4}$ with $k_{edge}=1.6327$, $k_{screw}=0.8435$,
$\sigma_{\gamma''} = M \max(\tau_{edge}, \tau_{screw})$.

**Solid solution:** $\sigma_{ss} = \sqrt{\sum_j \sigma_j^2}$, with $\sigma_j = K_j\sqrt{x_j}$ for solutes and the first
entry taken as $K_0\,(1 - \sum_k x_k)$.

All functions are vectorized over configurations; invalid inputs (e.g. zero volume fraction) propagate as NaN.

In [ ]:
def lattice_parameter(x_phase, elements=PHASE_ELEMENTS, lp=LATTICE_PARAM):
    # Vegard-type mixing rule with Ni as the balance element (Angstrom).
    coeffs = np.array([lp[e] for e in elements])
    return x_phase @ coeffs + (1.0 - x_phase.sum(axis=1)) * lp["Ni"]


def molar_volume(a_angstrom):
    # Molar volume (m^3/mol) of an FCC-type cell with 4 atoms.
    return N_A * 0.25 * a_angstrom**3 * 1e-30


def volume_fraction(f_ppt, vm_ppt, f_mat, vm_mat):
    # Precipitate volume fraction in a two-phase (matrix + precipitate) mixture.
    v_ppt = vm_ppt * f_ppt * 1e6   # cm^3
    v_mat = vm_mat * f_mat * 1e6   # cm^3
    return v_ppt / (v_mat + v_ppt)


def interparticle_spacing(vol_frac, r_bar):
    return (np.sqrt(np.pi / (4.0 * vol_frac)) - 1.0) * 2.0 * r_bar


def sigma_gamma_prime(vol_frac, r=R_GP, misfit=MISFIT_GP):
    # Coherency strengthening by spherical gamma', weakly coupled dislocations (MPa).
    r_bar = math.sqrt(2.0 / 3.0) * r
    lam = interparticle_spacing(vol_frac, r_bar)
    strain = (1.0 / 3.0) * ((1.0 + nu) / (1.0 - nu)) * misfit

    log_term = np.log(lam / (2.0 * b))
    pre = G * b**2 / (4.0 * math.pi)
    T_edge  = pre * ((1.0 + nu - 3.0 * nu * math.sin(math.pi / 2) ** 2) / (1.0 - nu)) * log_term
    T_screw = pre * (1.0 + nu - (3.0 * nu * math.sin(0.0) ** 2) / (1.0 - nu)) * log_term

    coh = G**3 * b * strain**3 * r_bar**3
    tau_edge  = 4.11 * np.sqrt(coh / (lam**2 * T_edge))
    tau_screw = 1.38 * np.sqrt(coh / (lam**2 * T_screw))
    return M * np.where(tau_edge > tau_screw, tau_edge, tau_screw) / 1e6


def sigma_gamma_double_prime(vol_frac, a=A_GDP, c=C_GDP, misfit=MISFIT_GDP):
    # Coherency strengthening by disc-shaped gamma'', strongly coupled dislocations (MPa).
    r_eq = (a**2 * c) ** (1.0 / 3.0)
    r_bar = math.sqrt(2.0 / 3.0) * r_eq
    lam = interparticle_spacing(vol_frac, r_bar)
    strain = (2.0 / 3.0) * misfit

    pre = G * b**2 / (4.0 * math.pi)
    T_edge  = pre * ((1.0 + nu - 3.0 * nu * math.sin(math.pi / 2) ** 2) / (1.0 - nu)) * math.log(LOG_R_GDP)
    T_screw = pre * ((1.0 + nu - 3.0 * nu * math.sin(0.0) ** 2) / (1.0 - nu)) * math.log(LOG_R_GDP)

    tau_edge  = 1.6327 * (0.8 / lam) * ((T_edge**3  * strain * G * r_bar) / b**3) ** 0.25
    tau_screw = 0.8435 * (0.8 / lam) * ((T_screw**3 * strain * G * r_bar) / b**3) ** 0.25
    return M * np.where(tau_edge > tau_screw, tau_edge, tau_screw) / 1e6


def sigma_solid_solution(x_matrix, k_ss=K_SS):
    # Solid-solution strengthening of the gamma matrix (MPa).
    k = np.asarray(k_ss, dtype=float)
    terms = k * np.sqrt(x_matrix)
    terms[:, 0] = k[0] * (1.0 - x_matrix.sum(axis=1))
    return np.sqrt((terms**2).sum(axis=1))

In [ ]:
with np.errstate(divide="ignore", invalid="ignore"):
    vm_matrix = molar_volume(lattice_parameter(x_matrix))
    vm_gp     = molar_volume(lattice_parameter(x_gp))
    vm_gdp    = molar_volume(lattice_parameter(x_gdp))

    vf_gp  = volume_fraction(f_gp,  vm_gp,  f_matrix, vm_matrix)
    vf_gdp = volume_fraction(f_gdp, vm_gdp, f_matrix, vm_matrix)

    s_gp  = sigma_gamma_prime(vf_gp)
    s_gdp = sigma_gamma_double_prime(vf_gdp)
    s_ss  = sigma_solid_solution(x_matrix)

ys = s_gp + s_gdp + SIGMA_0 + s_ss

results = pd.DataFrame(data["comp"], columns=BULK_ELEMENTS)
results["vf_gamma_prime_pct"] = 100 * vf_gp
results["vf_gamma_double_prime_pct"] = 100 * vf_gdp
results["sigma_gp_MPa"] = s_gp
results["sigma_gdp_MPa"] = s_gdp
results["sigma_ss_MPa"] = s_ss
results["YS"] = ys

n_nan = int(np.isnan(ys).sum())
print(f"Computed YS for {len(ys)} configurations ({n_nan} NaN).")
results.describe().T

## 6. Visualization

In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 10,
    "figure.dpi": 150,
    "axes.linewidth": 1.0,
})

valid = results.dropna(subset=["YS"])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(valid["YS"], bins=30, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Yield strength (MPa)")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of computed YS")
axes[0].grid(True, linestyle="--", alpha=0.5)

axes[1].scatter(valid["Nb"], valid["YS"], s=2, alpha=0.3)
axes[1].set_xlabel("Nb (bulk composition)")
axes[1].set_ylabel("Yield strength (MPa)")
axes[1].set_title("YS vs. Nb content")
plt.tight_layout()
plt.show()

### Optional: comparison with literature values
Provide `LITERATURE_CSV` with a single column `YS_MPa` containing reported room-temperature YS values
for the alloy family. The cell is skipped if the file is absent.

In [ ]:
if LITERATURE_CSV.exists():
    lit = pd.read_csv(LITERATURE_CSV)["YS_MPa"].dropna()

    fig, ax = plt.subplots(figsize=(110 / 25.4, 90 / 25.4))
    sns.histplot(x=valid["YS"], kde=True, bins=15, color="skyblue", edgecolor="black",
                 stat="density", label="Simulation", ax=ax)
    sns.histplot(x=lit, kde=True, bins=10, color="orange", edgecolor="black",
                 stat="density", alpha=0.5, label="Literature", ax=ax)
    ax.set_xlabel("YS (MPa)")
    ax.set_ylabel("Normalized frequency")
    ax.yaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    ax.ticklabel_format(style="sci", axis="y", scilimits=(0, 0))
    ax.legend(frameon=False)
    ax.set_title("Simulated vs. literature YS")
    plt.tight_layout()
    plt.show()
else:
    print(f"No literature file at {LITERATURE_CSV}; skipping comparison.")

## 7. Export (optional)
Writes the bulk composition and YS label table used for training. Do not commit outputs computed from
licensed Thermo-Calc data.

In [ ]:
if OUTPUT_PATH is not None:
    OUTPUT_PATH = Path(OUTPUT_PATH)
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    labels = results[BULK_ELEMENTS + ["YS"]].dropna()
    if OUTPUT_PATH.suffix == ".arrow":
        labels.reset_index(drop=True).to_feather(OUTPUT_PATH)
    else:
        labels.to_csv(OUTPUT_PATH, index=False)
    print(f"Saved {len(labels)} rows to {OUTPUT_PATH}")